In [1]:
# === Held-out evaluation for latest v3 run (macro/micro Dice, CSV+JSON) ===
from pathlib import Path
import importlib.util, json, time
import numpy as np

# --------- Paths (v3 run + held-out set) ----------
RUN_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813")
TEST_DIR  = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_lores")
T1_DIR    = TEST_DIR / "t1"      # use these to avoid duplicate pairing
MSK_DIR   = TEST_DIR / "masks"
TRAIN_MOD = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
# -----------------------------------------------

# ---- Import training module (utils + custom layers) ----
spec = importlib.util.spec_from_file_location("arc_seg_train", TRAIN_MOD)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# ---- Find model to load (prefer full .keras; else rebuild and load weights) ----
models_dir     = RUN_DIR / "models"
callbacks_dir  = RUN_DIR / "callbacks"
full_models    = sorted(models_dir.glob("*.keras"))
best_weights   = callbacks_dir / "best_model_dynamic.weights.h5"
cfg_json_path  = models_dir / "config.json"  # written by training code

model = None
INPUT_SHAPE = None

custom_objects = {
    "ResidualConvBlock": seg.ResidualConvBlock,
    "VisionMambaBlock": seg.VisionMambaBlock,
    "SAM2Attention": seg.SAM2Attention,
    "CombinedLoss": seg.CombinedLoss,
    "dice_coefficient": seg.dice_coefficient,
    "dice_loss": seg.dice_loss,
    "boundary_loss": seg.boundary_loss,
}

try:
    from keras.saving import load_model as keras_load_model
except Exception:
    from tensorflow.keras.models import load_model as keras_load_model

if full_models:
    model_path = full_models[-1]
    print(f"Loading FULL model: {model_path}")
    model = keras_load_model(model_path, compile=False, custom_objects=custom_objects)
    INPUT_SHAPE = tuple(model.input_shape[1:])
else:
    assert cfg_json_path.exists(), f"Missing {cfg_json_path}"
    with open(cfg_json_path) as f:
        saved_cfg = json.load(f)
    cfg = seg.DynamicTrainingConfig(
        DATA_DIR=TEST_DIR,  # dummy; not training now
        MODEL_DIR=models_dir,
        CALLBACKS_DIR=callbacks_dir,
        INPUT_SHAPE=tuple(saved_cfg["INPUT_SHAPE"]) if saved_cfg.get("INPUT_SHAPE") else None,
        BASE_FILTERS=int(saved_cfg.get("BASE_FILTERS", 8)),
        SAM_HEADS=int(saved_cfg.get("SAM_HEADS", 2)),
    )
    if cfg.INPUT_SHAPE in (None, (), []):
        raise RuntimeError("INPUT_SHAPE missing in saved config; cannot rebuild model.")
    INPUT_SHAPE = cfg.INPUT_SHAPE
    print("Rebuilding model from config.json and loading best weights…")
    model = seg.build_dynamic_model(cfg)
    model.load_weights(str(best_weights))

print("INPUT_SHAPE:", INPUT_SHAPE)

# ---- Build held-out list using separated subfolders (avoid duplicates) ----
if T1_DIR.exists() and MSK_DIR.exists():
    cfg_eval = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR, IMAGES_DIR=T1_DIR, MASKS_DIR=MSK_DIR,
                                         MODEL_DIR=RUN_DIR/"_tmp_models", CALLBACKS_DIR=RUN_DIR/"_tmp_callbacks")
else:
    cfg_eval = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR,
                                         MODEL_DIR=RUN_DIR/"_tmp_models", CALLBACKS_DIR=RUN_DIR/"_tmp_callbacks")

cfg_eval.INPUT_SHAPE = INPUT_SHAPE
pairs, lesion_presence = seg.load_generic_dataset(cfg_eval)
print(f"Pairs: {len(pairs)} | % non-empty masks: {lesion_presence.mean()*100:.1f}%")

# ---- Dice helpers ----
def dice_soft(y, p):
    y = y.astype(np.float64); p = p.astype(np.float64)
    inter = (y * p).sum()
    return (2.0*inter) / (y.sum() + p.sum() + 1e-12)

def dice_hard(y, p, th=0.5):
    pb = (p >= th).astype(np.float64)
    inter = (y*pb).sum()
    return (2.0*inter) / (y.sum() + pb.sum() + 1e-12)

# ---- Evaluate (macro: per-case mean; micro: global) ----
macro_softs, macro_hards = [], []
s_inter_soft = np.float64(0.0)
s_sumy_soft  = np.float64(0.0)
s_sump_soft  = np.float64(0.0)

TH = 0.50  # default; you can sweep later

for i, (img_p, msk_p) in enumerate(pairs, 1):
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    y   = seg._load_and_preprocess_mask(str(msk_p),  INPUT_SHAPE[:-1]).astype(np.float32)

    x = np.zeros((1,*INPUT_SHAPE), np.float32)
    x[0,...,0] = img
    p = model.predict(x, verbose=0)[0,...,0].astype(np.float32)

    ds = dice_soft(y, p)
    dh = dice_hard(y, p, th=TH)
    macro_softs.append(ds); macro_hards.append(dh)

    s_inter_soft += (y.astype(np.float64) * p.astype(np.float64)).sum()
    s_sumy_soft  += y.sum(dtype=np.float64)
    s_sump_soft  += p.sum(dtype=np.float64)

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] last case soft={ds:.4f} hard@{TH:.2f}={dh:.4f}")

macro_soft = float(np.mean(macro_softs))
macro_hard = float(np.mean(macro_hards))
micro_soft = float((2.0*s_inter_soft) / (s_sumy_soft + s_sump_soft + 1e-12))

# A second pass for exact micro-hard at the same threshold (memory-safe)
s_inter_hard = np.float64(0.0)
s_sumy_hard  = np.float64(0.0)
s_sump_hard  = np.float64(0.0)
for img_p, msk_p in pairs:
    y = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
    x = np.zeros((1,*INPUT_SHAPE), np.float32)
    x[0,...,0] = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    p = model.predict(x, verbose=0)[0,...,0]
    pb = (p >= TH).astype(np.float64)
    s_inter_hard += (y.astype(np.float64) * pb).sum()
    s_sumy_hard  += y.sum(dtype=np.float64)
    s_sump_hard  += pb.sum(dtype=np.float64)

micro_hard = float((2.0*s_inter_hard) / (s_sumy_hard + s_sump_hard + 1e-12))

# ---- Report + save ----
print("\n=== HELD-OUT RESULTS ===")
print(f"Per-case (macro) soft Dice       : {macro_soft:.4f}")
print(f"Per-case (macro) hard Dice @ {TH:.2f}: {macro_hard:.4f}")
print(f"Global (micro) soft Dice         : {micro_soft:.4f}")
print(f"Global (micro) hard Dice @ {TH:.2f} : {micro_hard:.4f}")
print(f"Val set size: {len(pairs)} cases")

out_dir = RUN_DIR / "test_eval"
out_dir.mkdir(parents=True, exist_ok=True)
ts = time.strftime("%Y%m%d_%H%M%S")

# per-case CSV
csv_path = out_dir / f"test_metrics_{ts}.csv"
with open(csv_path, "w") as f:
    f.write("case,soft_dice,hard_dice_at_{:.2f}\n".format(TH))
    for (img_p, _), ds, dh in zip(pairs, macro_softs, macro_hards):
        f.write(f"{img_p.stem},{ds:.6f},{dh:.6f}\n")

# summary JSON
summary = {
    "threshold": TH,
    "macro_soft": macro_soft,
    "macro_hard": macro_hard,
    "micro_soft": micro_soft,
    "micro_hard": micro_hard,
    "n_cases": len(pairs),
    "run_dir": str(RUN_DIR),
}
json_path = out_dir / f"test_metrics_summary_{ts}.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nWrote per-case CSV ->", csv_path)
print("Wrote summary JSON ->", json_path)


2025-11-10 16:04:48.098630: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1762815889.840159 1578934 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1762815889.841188 1578934 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21724 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1762815889.841528 1578934 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1762815889.842497 1578934 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 21701 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2025-11-10 16:04:49,905 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-10 16:04:49,906 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-10 16:04:49,906 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy
Loading FULL model: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/models/smart_sota_dynamic_20251110_101813.keras


2025-11-10 16:04:51,374 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-10 16:04:51,375 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.99GB | GPU mem tracking failed | Disk: 1230.2GB free
2025-11-10 16:04:51,382 - SmartSOTA_Dynamic - INFO - 📂 Two-folder mode: images=198 (/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_lores/t1), masks=198 (/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_lores/masks)
2025-11-10 16:04:51,383 - SmartSOTA_Dynamic - INFO - Found 198 image files and 198 mask files


INPUT_SHAPE: (192, 224, 192, 1)


2025-11-10 16:05:17,917 - SmartSOTA_Dynamic - INFO - 📊 Created 198 image–mask pairs
2025-11-10 16:05:17,918 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-11-10 16:05:17,919 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=1.00GB | GPU mem tracking failed | Disk: 1230.2GB free


Pairs: 198 | % non-empty masks: 100.0%


2025-11-10 16:05:18.976752: I external/local_xla/xla/service/service.cc:163] XLA service 0x7e76a80051e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-10 16:05:18.976780: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-11-10 16:05:18.976787: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (1): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-11-10 16:05:19.075946: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-10 16:05:19.350304: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91001
I0000 00:00:1762815925.605848 1579080 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


[10/198] last case soft=0.6928 hard@0.50=0.6929
[20/198] last case soft=0.7624 hard@0.50=0.7630
[30/198] last case soft=0.6775 hard@0.50=0.6792
[40/198] last case soft=0.5130 hard@0.50=0.5138
[50/198] last case soft=0.8952 hard@0.50=0.8955
[60/198] last case soft=0.0001 hard@0.50=0.0000
[70/198] last case soft=0.2955 hard@0.50=0.2945
[80/198] last case soft=0.6608 hard@0.50=0.6618
[90/198] last case soft=0.4297 hard@0.50=0.4304
[100/198] last case soft=0.7699 hard@0.50=0.7699
[110/198] last case soft=0.5387 hard@0.50=0.5380
[120/198] last case soft=0.8095 hard@0.50=0.8097
[130/198] last case soft=0.0000 hard@0.50=0.0000
[140/198] last case soft=0.7635 hard@0.50=0.7632
[150/198] last case soft=0.5915 hard@0.50=0.5951
[160/198] last case soft=0.7700 hard@0.50=0.7712
[170/198] last case soft=0.4189 hard@0.50=0.4221
[180/198] last case soft=0.0000 hard@0.50=0.0000
[190/198] last case soft=0.3326 hard@0.50=0.3329
[198/198] last case soft=0.6302 hard@0.50=0.6315

=== HELD-OUT RESULTS ===
Per